# 青木まりこ現象｜分析② ATUSの場所別活動時間

**研究目的**：書店以外の施設における活動時間を比較し、「書店の滞在時間が特別に長いから便意が起きるだけ」という説明を検討するための基礎資料を作る。

- データ：米国労働統計局（BLS）**American Time Use Survey (ATUS)** 公開個票（初期設定は **2023年**。2024・2025年はZIPを追加して実行）
- 原データ：[ATUS 公開データ一覧](https://www.bls.gov/tus/data.htm)
- 解析項目：`TUCASEID`（調査参加者）、`TEWHERE`（場所）、`TUACTDUR24`（活動時間［分］）、`TUFINLWGT`（調査ウェイト）
- 比較：図書館（10）、その他店舗・モール（7）、スーパー（6）、飲食店・バー（4）

> **研究上の重要な制約**：ATUSの「その他店舗・モール」には**書店が単独で分類されていない**。また、ATUSは「24時間日誌の活動時間」であり、**入店から退店までの厳密な滞在時間ではない**。まして便意は記録されていない。したがって結果から青木まりこ現象の原因や書店特有性を直接判定することはできない。

このノートブックは米国の統計データ（一次収集された公開個票を本研究で**二次分析**）をダウンロードし、集計、図示、CSV出力まで行う。日本の松田（2003）40人の研究とは標本・定義が異なるため、値を同一指標として直接比較しない。

In [ ]:
# 依存ライブラリ（Google Colab 標準環境向け）
import io, os, re, zipfile, base64
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

YEARS = [2023]  # 3年分のZIPを登録したら [2023, 2024, 2025] に変更
YEAR_LABEL = str(YEARS[0]) if len(YEARS) == 1 else f'{min(YEARS)}–{max(YEARS)}'

PROJECT_DIR = Path.cwd()
for folder in (PROJECT_DIR, *PROJECT_DIR.parents):
    if (folder / 'README.md').is_file() and (folder / 'notebooks').is_dir():
        PROJECT_DIR = folder
        break

DATA_DIR = PROJECT_DIR / 'data' / 'raw' / 'atus'
TABLE_DIR = PROJECT_DIR / 'results' / 'tables'
FIGURE_DIR = PROJECT_DIR / 'results' / 'figures'
NOTE_DIR = PROJECT_DIR / 'docs' / 'notes'
for directory in (DATA_DIR, TABLE_DIR, FIGURE_DIR, NOTE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CATEGORIES = {
    10: 'Library',
    7: 'Other store / mall',
    6: 'Grocery store',
    4: 'Restaurant / bar',
}

print('分析対象年:', YEARS)
print('比較カテゴリ:', CATEGORIES)

## 1. BLS公式データのダウンロード

各年の`Activity` ZIPと`Respondent` ZIPを読み込む。非公開GitHubの `data/atus/zips/` を最優先に参照する。Colabシークレットに `ATUS_GITHUB_TOKEN`（対象リポジトリに限定・Contents読み取り専用のトークン）を登録し、ノートブックからのアクセスを許可する。ColabからBLSへの接続が拒否された場合は、[BLSの年度別ページ](https://www.bls.gov/tus/data.htm)から該当ファイルをブラウザでダウンロードし、Colabへアップロードする。ZIPは展開不要。

In [ ]:
# 非公開GitHubの登録済みZIPを優先し、未登録ならBLS→手動アップロードにフォールバック
GITHUB_REPO = 'Roki0i/aoki-mariko-research'
GITHUB_ZIP_DIR = 'data/atus/zips'

def get_github_token():
    token = os.environ.get('ATUS_GITHUB_TOKEN', '').strip()
    if token:
        return token
    try:
        from google.colab import userdata
        return (userdata.get('ATUS_GITHUB_TOKEN') or '').strip()
    except Exception:
        return ''

def validate_atus_zip(content, year, kind):
    if not zipfile.is_zipfile(io.BytesIO(content)):
        raise ValueError('ZIP形式ではありません')
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        names = [Path(n).name.lower() for n in z.namelist()]
        expected = f'atus{kind}'
        if not any(n.startswith(expected) and str(year) in n
                   and n.endswith(('.csv', '.dat', '.txt')) for n in names):
            raise ValueError(f'{year}年の{kind}データがZIP内部に見つかりません: {names[:8]}')

def get_github_zip(year, kind):
    token = get_github_token()
    if not token:
        print('GitHub認証なし: Colabシークレット ATUS_GITHUB_TOKEN が未設定')
        return None
    filename = f'atus{kind}-{year}.zip'
    url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_ZIP_DIR}/{filename}'
    try:
        response = requests.get(
            url, params={'ref':'main'}, timeout=90,
            headers={
                'Authorization':f'Bearer {token}',
                'Accept':'application/vnd.github.raw+json',
                'X-GitHub-Api-Version':'2022-11-28',
            })
        response.raise_for_status()
        content = response.content
        if 'json' in response.headers.get('Content-Type', '').lower():
            obj = response.json()
            if obj.get('encoding') != 'base64':
                raise ValueError('GitHubがファイル本体を返しませんでした')
            content = base64.b64decode(obj['content'])
        validate_atus_zip(content, year, kind)
        print('GitHubから取得:', filename, f'({len(content):,} bytes)')
        return content
    except (requests.RequestException, ValueError, KeyError) as e:
        print('GitHubから取得できません:', filename, type(e).__name__, str(e)[:160])
        return None

def obtain_zip(year, kind):
    target = DATA_DIR / f'atus{kind}-{year}.zip'
    if target.exists():
        try:
            validate_atus_zip(target.read_bytes(), year, kind)
            print('ローカルキャッシュを使用:', target)
            return target
        except ValueError:
            target.unlink()

    github_data = get_github_zip(year, kind)
    if github_data is not None:
        target.write_bytes(github_data)
        return target

    url = f'https://www.bls.gov/tus/datafiles/atus{kind}-{year}.zip'
    print('BLS公式取得を試行:', url)
    try:
        response = requests.get(url, timeout=60, headers={
            'User-Agent':'Mozilla/5.0 (academic research; ATUS public microdata)',
            'Accept':'application/zip,application/octet-stream,*/*',
        })
        response.raise_for_status()
        validate_atus_zip(response.content, year, kind)
        target.write_bytes(response.content)
        print('BLSから取得:', target)
        return target
    except (requests.RequestException, ValueError) as e:
        print('BLS自動取得に失敗:', type(e).__name__, str(e)[:160])

    print('手動取得:', url)
    try:
        from google.colab import files
    except ImportError as e:
        raise RuntimeError(f'{target.name} を {DATA_DIR} に配置して再実行してください') from e
    print(f'{target.name} を選択してください（ほかのZIPは受け付けません）。')
    uploaded = files.upload()
    if target.name not in uploaded:
        raise RuntimeError(f'{target.name} がアップロードされませんでした')
    content = uploaded[target.name]
    validate_atus_zip(content, year, kind)
    target.write_bytes(content)
    return target

def read_zip_csv(path, columns, dtypes=None):
    with zipfile.ZipFile(path) as z:
        candidates = [n for n in z.namelist() if n.lower().endswith(('.csv', '.dat', '.txt'))]
        if not candidates:
            raise RuntimeError(f'{path} にCSV形式のファイルがありません。内容: {z.namelist()}')
        preferred = [n for n in candidates if re.search(r'atus(act|resp)', n, re.I)] or candidates
        filename = preferred[0]
        with z.open(filename) as f:
            data = pd.read_csv(f, encoding='latin1', low_memory=False,
                               dtype={'TUCASEID':str})
    data.columns = [str(c).strip().upper() for c in data.columns]
    missing = set(columns) - set(data.columns)
    if missing: raise RuntimeError(f'必要な列がありません：{missing} （実際の列例：{data.columns[:30].tolist()}）')
    return data[list(columns)].copy()

all_activity, all_respondent = [], []
for year in YEARS:
    apath = obtain_zip(year, 'act')
    rpath = obtain_zip(year, 'resp')
    a = read_zip_csv(apath, ['TUCASEID','TEWHERE','TUACTDUR24'])
    r = read_zip_csv(rpath, ['TUCASEID','TUFINLWGT'])
    a['year'] = r['year'] = year
    all_activity.append(a)
    all_respondent.append(r)
    print(year, 'activity rows =', len(a), 'respondents =', len(r))

activity = pd.concat(all_activity, ignore_index=True)
respondent = pd.concat(all_respondent, ignore_index=True)
assert not respondent[['year','TUCASEID']].duplicated().any(), 'respondent ID重複があります'
print('全活動行:', f'{len(activity):,}', '| 調査参加者:', f'{len(respondent):,}')

## 2. 場所別の活動時間を前処理

- 活動に付いた`TEWHERE`を施設コードとして使用する。
- `TUACTDUR24`は1日の各活動の時間［分］で、**訪問1回の滞在時間とは異なる**。
- **同一人物・同一日・同一場所コードの活動時間を合計**し、「その日その分類の場所で過ごした記録時間」として分析する。
- 調査ウェイトは`TUFINLWGT`を使用。複数年結合時は年数で割り、各年の人口を足し過ぎないようにする。

In [ ]:
# 基本的なバリデーションと除外件数を保存
activity['TEWHERE'] = pd.to_numeric(activity['TEWHERE'], errors='coerce')
activity['TUACTDUR24'] = pd.to_numeric(activity['TUACTDUR24'], errors='coerce')
respondent['TUFINLWGT'] = pd.to_numeric(respondent['TUFINLWGT'], errors='coerce')

original = len(activity)
chosen = activity[activity['TEWHERE'].isin(CATEGORIES)].copy()
chosen = chosen[chosen['TUACTDUR24'].between(0.01, 1440)].copy()
print('全活動数:', original, '| 対象の場所＆有効時間を持つ活動数:', len(chosen))

chosen['category'] = chosen['TEWHERE'].astype(int).map(CATEGORIES)
chosen['duration_min'] = chosen['TUACTDUR24']

# 連続訪問を厳密に識別できないため、同日・同場所分類で合計する
person_day = (chosen.groupby(['year','TUCASEID','category'], as_index=False)
              .agg(minutes=('duration_min','sum'),
                   activity_episodes=('duration_min','size')))
person_day = person_day.merge(respondent[['year','TUCASEID','TUFINLWGT']],
                              on=['year','TUCASEID'], how='left',validate='many_to_one')
person_day = person_day[(person_day['TUFINLWGT']>0)&person_day['TUFINLWGT'].notna()]
person_day['pool_weight'] = person_day['TUFINLWGT']/len(YEARS)

print('施設カテゴリ別の有効な個人×日レコード数：', len(person_day))
print(person_day.groupby('category').agg(person_days=('TUCASEID','size'), activities=('activity_episodes','sum')))
assert (person_day['minutes']>0).all()
assert (person_day['minutes']<=1440+1e-6).all(), '1日1440分を超過しています。データ確認が必要です。'

## 3. 調査ウェイトを使った集計

各カテゴリについて、**その場所で少なくとも1つの活動を記録した人**に限定した「1日あたり記録時間」を算出する。

- n：該当カテゴリーを訪れたと記録された回答者の日誌数（無加重）
- 加重平均：BLSの最終ウェイトを使用（複数年は年数で按分）
- 中央値：加重中央値・無加重中央値の両方を表示

※複数年でも1人につき1日の日誌。年度間で同じ個人を追跡するパネルではない。標本設計を反映した厳密な信頼区間・検定は本分析には含まない。

In [ ]:
def weighted_quantile(x, w, q):
    x = np.asarray(x, float); w = np.asarray(w, float)
    valid = np.isfinite(x)&np.isfinite(w)&(w>0)
    x, w = x[valid], w[valid]
    if len(x)==0: return np.nan
    order = np.argsort(x, kind='mergesort')
    x, w = x[order], w[order]
    return float(np.interp(q*sum(w), np.cumsum(w)-0.5*w, x))

records=[]
for code, cat in CATEGORIES.items():
    d = person_day[person_day['category']==cat]
    if len(d)==0: continue
    records.append({
        '場所':cat,
        '個人日誌数_n':len(d),
        '活動エピソード数':int(d['activity_episodes'].sum()),
        '非加重平均分':d['minutes'].mean(),
        '加重平均分':np.average(d['minutes'], weights=d['pool_weight']),
        '非加重中央値分':d['minutes'].median(),
        '加重中央値分':weighted_quantile(d['minutes'],d['pool_weight'],0.5),
        '加重75パーセンタイル分':weighted_quantile(d['minutes'],d['pool_weight'],0.75),
        '加重90パーセンタイル分':weighted_quantile(d['minutes'],d['pool_weight'],0.90),
        '最大分':d['minutes'].max()
    })
summary = pd.DataFrame(records)
pd.set_option('display.float_format',lambda v:f'{v:,.2f}')
print(f'対象：{YEAR_LABEL}年、米国ATUS')
print('指標：1人1日・場所分類別の活動時間（訪問ごとの厳密滞在時間ではない）')
display(summary)

## 4. 分布を可視化

全記録の分布を比較する。横軸は該当分類で過ごした**日誌上の合計分数**。箱ひげ図では外れ値の点表示を省略する（数値の集計には上限を適用しない）。

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
order = list(CATEGORIES.values())
labels, vals=[] , []
for cat in order:
    d = person_day[person_day.category==cat]['minutes'].to_numpy()
    if len(d):
        labels.append(cat+f' (n={len(d):,})')
        vals.append(d)
ax.boxplot(vals, vert=False, tick_labels=labels, showfliers=False)
ax.set_xlabel('Minutes per diary day in location category')
ax.set_title(f'ATUS {YEAR_LABEL}: daily activity minutes by location')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'atus_location_boxplot.png',dpi=180)
plt.show()

fig,ax=plt.subplots(figsize=(9,4.8))
plot=summary.sort_values('加重平均分')
ax.barh(plot['場所'], plot['加重平均分'])
for i, v in enumerate(plot['加重平均分']):
    ax.text(v+1,i,f'{v:.1f} min',va='center')
ax.set_xlim(0,max(plot['加重平均分'])*1.3)
ax.set_xlabel('Weighted mean minutes per diary day (among observed visitors)')
ax.set_title(f'ATUS {YEAR_LABEL} | location categories')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'atus_weighted_means.png',dpi=180)
plt.show()

## 5. 保存と中間報告用の解釈

- CSVは**分析済みの加工データ**。BLSの元ZIPは上の`atus_raw`に残る。
- データを研究に使う際には、**米国の分析対象年（実行時のYEARS）、活動時間の記録、書店を単独識別できない、便意は未測定**と必ず説明する。
- 施設別の値は「その日に分類された場所で行った活動の総時間」。同日に複数訪問した場合も合算され、店舗間移動も区別できない。
- この分析は**施設滞在の背景比較**であり、青木まりこ現象そのものの実証ではない。

In [ ]:
summary.to_csv(TABLE_DIR / 'atus_location_summary.csv',index=False,encoding='utf-8-sig')
person_day.to_csv(DATA_DIR / 'atus_person_day_selected.csv',index=False,encoding='utf-8-sig')

lines = [
    '# ATUS場所別活動時間・分析結果',
    f'- 対象年: {YEAR_LABEL}',
    f'- 元活動レコード数: {original:,}',
    f'- 対象の場所・有効時間を持つ活動数: {len(chosen):,}',
    f'- 場所別の個人日誌レコード数: {len(person_day):,}',
    '- 指標: 場所分類別の1日合計活動時間。1訪問あたりの滞在時間ではない。',
    '- 書店単独の識別不可、便意変数なし。原因仮説の確証には使用できない。',
    '',
    '## 加重平均（分）',
]
lines.extend(f"- {r['場所']}: n={int(r['個人日誌数_n']):,}, 加重平均={r['加重平均分']:.2f}分, 加重中央値={r['加重中央値分']:.2f}分" for _,r in summary.iterrows())
(NOTE_DIR / 'atus_colab_report.md').write_text('\n'.join(lines),encoding='utf-8')
print('集計表:', TABLE_DIR / 'atus_location_summary.csv')
print('元データ・個人日誌データ:', DATA_DIR)
print('図表:', FIGURE_DIR)
print('分析メモ:', NOTE_DIR / 'atus_colab_report.md')

# ダウンロード（必要な場合のみ実行）
# from google.colab import files
# files.download(str(TABLE_DIR / 'atus_location_summary.csv'))